# Assignment 7 - Baseline models

Short, reproducible notebook for baseline models. The logic remains aligned with Assignments 1-6: target `hospital_use_per_1000`, F0/F1 feature sets, seed 42, PracticeCode-aware split, R2/RMSE/MAE, and no preprocessing fitted outside training folds.

In [1]:
from pathlib import Path
import json, math, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED, TEST_SIZE, CV_SPLITS = 42, 0.2, 5
TARGET, GROUP = "hospital_use_per_1000", "PracticeCode"
ROOT = Path.cwd()
if ROOT.name == "assignment7":
    ROOT = ROOT.parent

def clean_json(value):
    if isinstance(value, dict):
        return {str(k): clean_json(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_json(v) for v in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    return value

In [2]:
raw_dir = ROOT / "assignment4" / "data" / "raw"
dem_path = raw_dir / "sample_gp_practice_population_demographics.csv"
sup_path = raw_dir / "sample_gp_practice_supporting_inputs.csv"
dem = pd.read_csv(dem_path, parse_dates=["Date"])
sup = pd.read_csv(sup_path, parse_dates=["Date"])
dem[GROUP] = dem[GROUP].astype(str)
sup[GROUP] = sup[GROUP].astype(str)

age_cols = ["AllAges", "Ages0to4", "Ages5to14", "Ages15to24", "Ages25to44", "Ages45to64", "Ages65to74", "Ages75to84", "Ages85plus"]
dem[age_cols] = dem[age_cols].apply(pd.to_numeric, errors="coerce")
keys = ["Date", GROUP]
valid_keys = dem.groupby(keys)["Sex"].agg(lambda s: {"All", "Female"}.issubset(set(s)))
valid_keys = valid_keys[valid_keys].index

base = dem.set_index(keys).loc[valid_keys].reset_index()
all_rows = base[base["Sex"].eq("All")].copy()
female = base[base["Sex"].eq("Female")][keys + ["AllAges"]].rename(columns={"AllAges": "female_population"})
df = all_rows[keys + ["HB", "HSCP"] + age_cols].merge(female, on=keys, validate="1:1")
df = df[df["AllAges"].fillna(0).gt(0)].copy()
df["share_age_0_14"] = (df["Ages0to4"] + df["Ages5to14"]) / df["AllAges"]
df["share_age_65_plus"] = (df["Ages65to74"] + df["Ages75to84"] + df["Ages85plus"]) / df["AllAges"]
df["share_female"] = df["female_population"] / df["AllAges"]
df["log_all_ages"] = np.log1p(df["AllAges"])
df = df.merge(sup[keys + ["gp_availability", "deprivation_index", TARGET]], on=keys, validate="1:1")
df = df.sort_values(["Date", GROUP]).reset_index(drop=True)
df["Date"] = df["Date"].dt.strftime("%Y-%m-%d")
df.head()

,Date,PracticeCode,HB,HSCP,AllAges,Ages0to4,Ages5to14,Ages15to24,Ages25to44,Ages45to64,...,Ages75to84,Ages85plus,female_population,share_age_0_14,share_age_65_plus,share_female,log_all_ages,gp_availability,deprivation_index,hospital_use_per_1000
0,2024-01-01,1001,NHS_Lothian,Edinburgh_North,5400,260,580,720,1480,1360,...,310,130,2790,0.155556,0.185185,0.516667,8.594339,0.68,18.2,268.4
1,2024-01-01,1002,NHS_Lothian,Edinburgh_West,4300,210,470,560,1170,1100,...,230,90,2235,0.158140,0.183721,0.519767,8.366603,0.63,24.9,286.7
2,2024-01-01,1003,NHS_Greater_Glasgow,Glasgow_Central,6100,340,760,840,1710,1500,...,250,90,3205,0.180328,0.155738,0.525410,8.716208,0.51,41.3,347.9
3,2024-01-01,1004,NHS_Grampian,Aberdeen_City,3900,180,420,500,990,1010,...,240,90,2015,0.153846,0.205128,0.516667,8.268988,0.73,20.8,248.6
4,2024-02-01,1001,NHS_Lothian,Edinburgh_North,5425,262,582,724,1486,1368,...,311,130,2802,0.155576,0.184885,0.516498,8.598957,0.69,18.1,267.2


In [3]:
rng = np.random.default_rng(SEED)
groups = np.array(sorted(df[GROUP].unique()))
test_n = max(1, math.ceil(TEST_SIZE * len(groups)))
test_groups = set(rng.permutation(groups)[:test_n])
is_test = df[GROUP].isin(test_groups)
train, test = df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)

fold_count = min(CV_SPLITS, train[GROUP].nunique())
fold_groups = np.array_split(np.random.default_rng(SEED).permutation(sorted(train[GROUP].unique())), fold_count)
folds = [(train[~train[GROUP].isin(g)].copy(), train[train[GROUP].isin(g)].copy()) for g in fold_groups]
sorted(test_groups), fold_count

(['1004'], 3)

In [4]:
def design(fit_df, apply_df, numeric, categorical):
    med = fit_df[numeric].median()
    x_fit = fit_df[numeric].fillna(med).astype(float)
    mean, std = x_fit.mean(), x_fit.std(ddof=0).replace(0, 1)
    parts = [((apply_df[numeric].fillna(med).astype(float) - mean) / std).reset_index(drop=True)]
    for col in categorical:
        mode = fit_df[col].mode(dropna=True)
        mode = mode.iloc[0] if len(mode) else "__missing__"
        levels = sorted(fit_df[col].fillna(mode).astype(str).unique())
        cat = pd.Series(pd.Categorical(apply_df[col].fillna(mode).astype(str), categories=levels), name=col)
        parts.append(pd.get_dummies(cat, prefix=col).astype(float).reset_index(drop=True))
    x = pd.concat(parts, axis=1)
    return x.to_numpy(float), list(x.columns)

def metrics(y, pred):
    err = y - pred
    denom = float(((y - y.mean()) ** 2).sum())
    return {"r2": round(1 - float((err ** 2).sum()) / denom, 6) if denom else None,
            "rmse": round(math.sqrt(float((err ** 2).mean())), 6),
            "mae": round(float(np.abs(err).mean()), 6)}

def fit_linear(x, y, alpha=0.0):
    z = np.c_[np.ones(len(x)), x]
    penalty = np.eye(z.shape[1]) * alpha
    penalty[0, 0] = 0
    return np.linalg.pinv(z.T @ z + penalty) @ z.T @ y

def predict_linear(beta, x):
    return np.c_[np.ones(len(x)), x] @ beta

def fit_tree(x, y, max_depth=3, min_leaf=2):
    def build(x, y, depth):
        pred = float(y.mean())
        if depth >= max_depth or len(y) < 2 * min_leaf or np.allclose(y, y[0]):
            return (pred, None, None, None, None)
        best = None
        for j in range(x.shape[1]):
            vals = np.unique(x[:, j])
            for t in (vals[:-1] + vals[1:]) / 2:
                left = x[:, j] <= t
                if left.sum() >= min_leaf and (~left).sum() >= min_leaf:
                    loss = ((y[left] - y[left].mean()) ** 2).sum() + ((y[~left] - y[~left].mean()) ** 2).sum()
                    if best is None or loss < best[0]:
                        best = (loss, j, t, left)
        if best is None:
            return (pred, None, None, None, None)
        _, j, t, left = best
        return (pred, j, t, build(x[left], y[left], depth + 1), build(x[~left], y[~left], depth + 1))
    return build(x, y, 0)

def predict_tree(tree, x):
    def one(row, node):
        pred, j, t, left, right = node
        return pred if j is None else one(row, left if row[j] <= t else right)
    return np.array([one(row, tree) for row in x])

In [5]:
F0 = ["log_all_ages", "gp_availability"]
F1 = F0 + ["share_age_65_plus", "share_female", "deprivation_index"]
models = [
    dict(name="dummy_mean_f0", display_name="Dummy mean baseline", family="dummy", feature_set="F0", num=F0, cat=[], params=[{}], strategy="No tuning; predicts the training-set mean."),
    dict(name="ols_f0", display_name="OLS baseline", family="linear", feature_set="F0", num=F0, cat=[], params=[{"alpha": 0.0}], strategy="No tuning; ordinary least squares."),
    dict(name="ridge_f1", display_name="Ridge extended baseline", family="linear", feature_set="F1", num=F1, cat=["HB", "HSCP"], params=[{"alpha": a} for a in [0.1, 1.0, 10.0]], strategy="Limited alpha grid selected by group-aware CV on training folds only."),
    dict(name="decision_tree_f1", display_name="Decision Tree benchmark", family="tree", feature_set="F1", num=F1, cat=["HB", "HSCP"], params=[{"max_depth": 3, "min_leaf": 2}], strategy="No tuning; conservative tree depth and leaf size fixed before test evaluation."),
]

def evaluate(spec, tr, te):
    candidates = []
    for params in spec["params"]:
        fold_scores = []
        for ftr, fva in folds:
            xtr, _ = design(ftr, ftr, spec["num"], spec["cat"])
            xva, _ = design(ftr, fva, spec["num"], spec["cat"])
            ytr, yva = ftr[TARGET].to_numpy(float), fva[TARGET].to_numpy(float)
            if spec["family"] == "dummy":
                pred = np.repeat(ytr.mean(), len(yva))
            elif spec["family"] == "linear":
                pred = predict_linear(fit_linear(xtr, ytr, params["alpha"]), xva)
            else:
                pred = predict_tree(fit_tree(xtr, ytr, **params), xva)
            fold_scores.append(metrics(yva, pred))
        cv = {"folds": len(fold_scores)}
        for m in ["r2", "rmse", "mae"]:
            vals = np.array([s[m] for s in fold_scores if s[m] is not None], float)
            cv[f"mean_{m}"] = round(float(vals.mean()), 6) if len(vals) else None
            cv[f"std_{m}"] = round(float(vals.std(ddof=1)), 6) if len(vals) > 1 else 0.0
        candidates.append({"params": params, "cv_summary": cv, "fold_metrics": fold_scores})
    selected = min(candidates, key=lambda c: c["cv_summary"]["mean_rmse"])

    xtr, cols = design(tr, tr, spec["num"], spec["cat"])
    xte, _ = design(tr, te, spec["num"], spec["cat"])
    ytr, yte = tr[TARGET].to_numpy(float), te[TARGET].to_numpy(float)
    if spec["family"] == "dummy":
        pred = np.repeat(ytr.mean(), len(yte))
    elif spec["family"] == "linear":
        pred = predict_linear(fit_linear(xtr, ytr, selected["params"]["alpha"]), xte)
    else:
        pred = predict_tree(fit_tree(xtr, ytr, **selected["params"]), xte)
    pred_rows = te[["Date", GROUP, TARGET]].copy()
    pred_rows["model_name"] = spec["name"]
    pred_rows["prediction"] = pred.round(6)
    pred_rows["residual"] = (pred_rows[TARGET] - pred_rows["prediction"]).round(6)
    return {"model_name": spec["name"], "display_name": spec["display_name"], "family": spec["family"],
            "feature_set": spec["feature_set"], "selected_params": selected["params"], "tuning_strategy": spec["strategy"],
            "encoded_feature_count": len(cols), "holdout": metrics(yte, pred), "cross_validation": selected["cv_summary"],
            "candidate_results": candidates}, pred_rows

In [6]:
results, predictions = zip(*(evaluate(spec, train, test) for spec in models))
rows = []
for r in results:
    rows.append({"model_name": r["model_name"], "display_name": r["display_name"],
                 "family": r["family"], "feature_set": r["feature_set"],
                 "selected_params": json.dumps(r["selected_params"], sort_keys=True),
                 "tuning_strategy": r["tuning_strategy"],
                 "encoded_feature_count": r["encoded_feature_count"],
                 **{f"holdout_{k}": v for k, v in r["holdout"].items()},
                 **{f"cv_{k}": v for k, v in r["cross_validation"].items()}})
metrics_table = pd.DataFrame(rows)
ols = metrics_table[metrics_table.model_name.eq("ols_f0")].iloc[0]
comparison = metrics_table.copy()
comparison["delta_r2_vs_ols_f0"] = comparison["holdout_r2"] - ols["holdout_r2"]
comparison["rmse_reduction_pct_vs_ols_f0"] = (
    (ols["holdout_rmse"] - comparison["holdout_rmse"]) / ols["holdout_rmse"] * 100
).round(6)
comparison = comparison.sort_values(["holdout_rmse", "model_name"]).reset_index(drop=True)
comparison[["display_name", "feature_set", "selected_params",
             "holdout_r2", "holdout_rmse", "holdout_mae",
             "cv_mean_rmse", "cv_std_rmse",
             "delta_r2_vs_ols_f0", "rmse_reduction_pct_vs_ols_f0"]]

In [ ]:
# Model comparison — RMSE bar charts
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

model_labels = comparison["display_name"].tolist()
bar_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

axes[0].bar(range(len(model_labels)), comparison["holdout_rmse"].tolist(),
            color=bar_colors[:len(model_labels)])
axes[0].set_xticks(range(len(model_labels)))
axes[0].set_xticklabels(model_labels, rotation=20, ha="right", fontsize=9)
axes[0].set_ylabel("RMSE")
axes[0].set_title("Holdout RMSE by Model")

axes[1].bar(range(len(model_labels)), comparison["cv_mean_rmse"].tolist(),
            color=bar_colors[:len(model_labels)])
axes[1].set_xticks(range(len(model_labels)))
axes[1].set_xticklabels(model_labels, rotation=20, ha="right", fontsize=9)
axes[1].set_ylabel("RMSE")
axes[1].set_title("CV Mean RMSE by Model")

plt.tight_layout()
plt.show()

# Actual vs Predicted for OLS F0 and Ridge F1
all_preds = pd.concat(predictions, ignore_index=True)
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5))

for ax2, (spec_name, label) in zip(axes2, [("ols_f0", "OLS F0"), ("ridge_f1", "Ridge F1")]):
    subset = all_preds[all_preds["model_name"] == spec_name]
    y_true = subset[TARGET].values
    y_pred = subset["prediction"].values
    ax2.scatter(y_true, y_pred, alpha=0.8)
    lo = min(y_true.min(), y_pred.min()) - 5
    hi = max(y_true.max(), y_pred.max()) + 5
    ax2.plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="Perfect prediction")
    ax2.set_xlabel("Actual hospital_use_per_1000")
    ax2.set_ylabel("Predicted")
    ax2.set_title(f"Actual vs Predicted \u2014 {label}")
    ax2.legend()

plt.tight_layout()
plt.show()